In [ ]:
class Character:
    def __init__(self, name, role, trait):
        self.name = name
        self.role = role
        self.trait = trait


class PlayerState:
    def __init__(self, name):
        self.name = name
        self.energy = 3
        self.memory_sync = 0
        self.trust_akali = 0
        self.trust_katarina = 0
        self.trust_sett = 0
        self.inventory = []
        self.clues = set()
        self.route = []
        self.ending = ''


class NorthLakeNovel:
    def __init__(self):
        self.characters = [
            Character('Сетт', 'старший вожатый', 'харизматичный и опасный наблюдатель'),
            Character('Акали', 'исследовательница', 'тихая, внимательная, ведёт дневник'),
            Character('Катарина', 'авантюристка', 'резкая, решительная, ищет артефакты')
        ]

        self.map_parts = {
            'водонапорная башня': 'ржавая пластина с координатами лесной тропы',
            'медпункт': 'лист с пометкой «Архив Смен: питание отключено»',
            'клуб': 'фрагмент карты с номером сектора 17',
            'сцена лагеря': 'кассета с пометкой «воспоминания смены»'
        }

        self.fixed_lore = {
            'лагерь': 'Лагерь «Северное Озеро» каждый год закапывает капсулу времени.',
            'тайна': 'Капсула №17 хранит не предметы, а воспоминания.',
            'архив': 'Архив Смен — машина, смешивающая память разных поколений.'
        }

        self.state = None

    def custom_print(self, text):
        print(text)

    def ask(self, prompt):
        self.custom_print(prompt)
        return input('> ').strip()

    def show_inventory(self):
        self.custom_print('')
        self.custom_print('=== ИНВЕНТАРЬ ===')
        if len(self.state.inventory) == 0:
            self.custom_print('Пока пусто.')
        else:
            for item in self.state.inventory:
                self.custom_print('- ' + item)
        self.custom_print('=================')

    def choose(self, prompt, options):
        self.custom_print('')
        self.custom_print(prompt)
        for key in options:
            self.custom_print(str(key) + '. ' + options[key])
        self.custom_print("(Введи 'i', чтобы открыть инвентарь)")

        answer = input('> ').strip().lower()
        while answer not in options:
            if answer == 'i':
                self.show_inventory()
            else:
                self.custom_print('Нужно ввести номер из списка.')
            answer = input('> ').strip().lower()
        return answer

    def prologue(self):
        name = self.ask('Имя героя (Артём/Алексей/Данила/Кирилл или любое другое):')
        if name == '':
            name = 'Артём'
        self.state = PlayerState(name)

        self.custom_print('')
        self.custom_print('Зимний вечер. Автобус режет туман фарами.')
        self.custom_print('Тебе снится лагерь, в котором ты будто уже был.')
        self.custom_print('Ты видишь костёр, старую сцену и голос: «Смена №∞ уже началась».')
        self.custom_print('Просыпаешься у ворот лагеря «Северное Озеро».')

        self.state.memory_sync += 1
        self.state.route.append('Пролог: сон о лагере')
        self.state.clues.add('дежавю у ворот')
        self.state.inventory.append('старый пропуск без даты')

    def chapter1_new_shift(self):
        self.custom_print('')
        self.custom_print('Глава 1 — Новая смена')
        self.custom_print(self.fixed_lore['лагерь'])

        for ch in self.characters:
            self.custom_print('- ' + ch.name + ': ' + ch.trait)

        choice = self.choose(
            'Кому помочь в первый день?',
            {'1': 'Акали (поиск старых записей)', '2': 'Катарине (поиск карты территории)'}
        )

        if choice == '1':
            self.state.trust_akali += 2
            self.state.memory_sync += 1
            self.state.inventory.append('дневник наблюдений Акали')
            self.state.route.append('Помог Акали')
            self.custom_print('Акали даёт тебе копию заметок о закрытых зонах лагеря.')
        else:
            self.state.trust_katarina += 2
            self.state.courage = 1
            self.state.inventory.append('компас Катарины')
            self.state.route.append('Помог Катарине')
            self.custom_print('Катарина находит тайную тропу и делится координатами.')

    def chapter2_old_camp(self):
        self.custom_print('')
        self.custom_print('Глава 2 — Старый лагерь')
        self.custom_print('Вы находите сгоревший корпус №3 и обугленные документы.')

        action = self.choose(
            'Что сделать с документами?',
            {'1': 'Передать Акали для анализа', '2': 'Спрятать и показать только Катарине', '3': 'Отнести Сетту'}
        )

        if action == '1':
            self.state.trust_akali += 1
            self.state.clues.add('протокол эксперимента')
            self.state.route.append('Документы у Акали')
        elif action == '2':
            self.state.trust_katarina += 1
            self.state.clues.add('карта корпуса №3')
            self.state.route.append('Документы у Катарины')
        else:
            self.state.trust_sett += 2
            self.state.energy += 1
            self.state.route.append('Документы у Сетта')
            self.custom_print('Сетт смотрит долго и говорит: «Иногда лучше не копаться в прошлом».')

        self.state.inventory.append('обрывок карты смены №17')

    def chapter3_map_parts(self):
        self.custom_print('')
        self.custom_print('Глава 3 — Части карты')
        self.custom_print('Карта разделена на 4 части. Нужно собрать все.')

        remaining = list(self.map_parts.keys())
        while len(remaining) > 0:
            self.custom_print('')
            self.custom_print('Осталось мест: ' + ', '.join(remaining))
            self.custom_print("(Или введи 'i' для инвентаря)")
            place = input('Куда идём дальше? > ').strip().lower()

            if place == 'i':
                self.show_inventory()
                continue

            normalized = None
            for candidate in remaining:
                if place == candidate.lower():
                    normalized = candidate
                    break

            if normalized is None:
                self.custom_print('Такого места в текущем списке нет.')
                self.state.energy -= 1
                continue

            self.custom_print('Найдено: ' + self.map_parts[normalized])
            self.state.inventory.append('фрагмент: ' + normalized)
            self.state.clues.add('фрагмент карты: ' + normalized)
            self.state.route.append('Посетил: ' + normalized)
            remaining.remove(normalized)

        self.state.memory_sync += 1

    def chapter4_dreams(self):
        self.custom_print('')
        self.custom_print('Глава 4 — Странные сны')
        self.custom_print('Тебе снятся дети у костра и лагерь 90-х, хотя ты не мог это помнить.')

        ans = self.choose(
            'Как реагировать на сны?',
            {'1': 'Записать всё в дневник', '2': 'Игнорировать', '3': 'Рассказать Сетту'}
        )

        if ans == '1':
            self.state.memory_sync += 2
            self.state.inventory.append('тетрадь со снами')
            self.state.clues.add('совпадения из снов')
            self.state.route.append('Анализировал сны')
        elif ans == '2':
            self.state.energy += 1
            self.state.route.append('Игнорировал сны')
        else:
            self.state.trust_sett += 1
            self.state.route.append('Поделился снами с Сеттом')
            self.custom_print('Сетт хмурится: «Если найдёшь Архив, не включай его».')

    def chapter5_sett_interferes(self):
        self.custom_print('')
        self.custom_print('Глава 5 — Сетт вмешивается')

        lock = self.choose(
            'Часть территории закрыли. Что делать?',
            {'1': 'Обойти запрет через лес', '2': 'Попросить Сетта открыть доступ', '3': 'Ждать ночи'}
        )

        if lock == '1':
            self.state.energy -= 1
            self.state.courage = self.state.courage + 1
            self.state.route.append('Обошёл запрет через лес')
        elif lock == '2':
            self.state.trust_sett += 1
            self.state.route.append('Договорился с Сеттом')
        else:
            self.state.memory_sync += 1
            self.state.route.append('Пошёл ночью')

    def chapter6_restricted_zone(self):
        self.custom_print('')
        self.custom_print('Глава 6 — Запретная зона')
        self.custom_print('Собранная карта ведёт к бетонному бункеру в лесу.')
        self.custom_print(self.fixed_lore['тайна'])
        self.custom_print(self.fixed_lore['архив'])

        self.state.clues.add('бункер архива')
        self.state.inventory.append('ключ сектора 17')
        self.state.route.append('Нашёл бункер Архива')

    def chapter7_final_choice(self):
        self.custom_print('')
        self.custom_print('Глава 7 — Финальный выбор')

        final = self.choose(
            'Что делать с Архивом Смен?',
            {
                '1': 'Поддержать Акали: раскрыть правду и закрыть Архив',
                '2': 'Поддержать Катарину: забрать устройство',
                '3': 'Поддержать Сетта: оставить всё как есть',
                '4': 'Перезаписать систему добровольной памятью (риск)'
            }
        )

        self.state.route.append('Финальный выбор: ' + final)

        if final == '1':
            self.state.ending = 'Концовка Акали: правда раскрыта, Архив уничтожен, лагерь закрывают.'
        elif final == '2':
            self.state.ending = 'Концовка Катарины: Архив увозят, воспоминания лагеря постепенно исчезают.'
        elif final == '3':
            self.state.ending = 'Концовка Сетта: Архив остаётся скрытым, лагерь продолжает существовать.'
        else:
            can_true = (
                self.state.memory_sync >= 4 and
                self.state.trust_akali >= 1 and
                self.state.trust_katarina >= 1 and
                'бункер архива' in self.state.clues
            )
            if can_true:
                self.state.ending = 'Истинная концовка: ты понимаешь, что Архив создан из твоей памяти, и перезаписываешь систему добровольного сохранения воспоминаний.'
            else:
                self.state.ending = 'Сломанная перезапись: система повреждена, а часть воспоминаний лагеря теряется безвозвратно.'

    def build_report(self):
        report = {}
        report['герой'] = self.state.name
        report['энергия'] = self.state.energy
        report['синхронизация_памяти'] = self.state.memory_sync
        report['доверие_акали'] = self.state.trust_akali
        report['доверие_катарина'] = self.state.trust_katarina
        report['доверие_сетт'] = self.state.trust_sett
        report['маршрут'] = self.state.route
        report['улики'] = list(self.state.clues)
        report['инвентарь'] = self.state.inventory
        report['концовка'] = self.state.ending
        return report

    def save_report(self, report):
        lines = []
        lines.append('Герой: ' + report['герой'])
        lines.append('Энергия: ' + str(report['энергия']))
        lines.append('Синхронизация памяти: ' + str(report['синхронизация_памяти']))
        lines.append('Доверие Акали: ' + str(report['доверие_акали']))
        lines.append('Доверие Катарины: ' + str(report['доверие_катарина']))
        lines.append('Доверие Сетта: ' + str(report['доверие_сетт']))
        lines.append('Маршрут:')
        for step in report['маршрут']:
            lines.append('- ' + step)
        lines.append('Улики: ' + ', '.join(report['улики']))
        lines.append('Инвентарь: ' + ', '.join(report['инвентарь']))
        lines.append('Финал: ' + report['концовка'])

        with open('novel_result.txt', 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))

    def run(self):
        self.prologue()
        self.chapter1_new_shift()
        self.chapter2_old_camp()
        self.chapter3_map_parts()
        self.chapter4_dreams()
        self.chapter5_sett_interferes()
        self.chapter6_restricted_zone()
        self.chapter7_final_choice()

        report = self.build_report()
        self.save_report(report)

        self.custom_print('')
        self.custom_print(self.state.ending)
        self.custom_print('Итог сохранён в novel_result.txt')


novel = NorthLakeNovel()
novel.run()
